# 실습 3: MLflow 실험 관리 심화

**목표**: MLflow로 ML 실험을 체계적으로 기록하고 관리하는 방법을 익힙니다.

**3파트 구성**:
- Part A: 자동 로깅의 마법 (`autolog`)
- Part B: 커스텀 로깅 (파라미터, 메트릭, 아티팩트)
- Part C: 모델 시그니처 & 입력 예제

**소요 시간**: ~50분

## Part A: 자동 로깅의 마법 (20분)

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd

# 데이터 준비
CATALOG = "3dt005_databricks"
SCHEMA = "wine"

wine_df = spark.table(f"{CATALOG}.{SCHEMA}.wine_quality_lab").toPandas()
X = wine_df.drop(["is_good_quality", "quality"], axis=1, errors="ignore")
y = wine_df["is_good_quality"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"학습: {len(X_train)}개, 테스트: {len(X_test)}개")

학습: 3918개, 테스트: 980개


### autolog() — 코드 한 줄의 마법
Databricks에서는 기본 활성화되어 있지만, 명시적으로 호출해봅니다.

In [0]:
# 자동 로깅 활성화
mlflow.autolog()

# 실험 이름 설정
mlflow.set_experiment("/Users/" + spark.sql("SELECT current_user()").first()[0] + "/wine_quality_mlflow_lab")

# 모델 학습 — autolog이 자동으로 모든 것을 기록합니다!
with mlflow.start_run(run_name="autolog_rf_baseline"):
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

2026/03/24 02:06:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/24 02:06:47 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/03/24 02:06:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/03/24 02:06:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.ml.
2026/03/24 02:06:49 INFO mlflow.tracking.fluent: Experiment with name '/Users/3dt005@msacademy.msai.kr/wine_quality_mlflow_lab' does not exist. Creating a new experiment.


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

2026/03/24 02:06:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils."


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

F1 Score: 0.4254
Accuracy: 0.8153


💡 **Experiments UI에서 확인해보세요!**
- 왼쪽 메뉴 → Experiments → 방금 생성된 실험 클릭
- 자동 기록된 항목: 하이퍼파라미터, 메트릭, 학습된 모델, Feature Importance

### 여러 모델 비교 실험

In [0]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "rf_shallow": RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42),
    "rf_deep": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    "gb_default": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "lr_baseline": LogisticRegression(max_iter=1000, random_state=42),
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        print(f"{name}: F1={f1_score(y_test, y_pred):.4f}")

Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

rf_shallow: F1=0.1868


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

...

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

gb_default: F1=0.5420


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

lr_baseline: F1=0.3519


💡 **Experiments UI에서 런을 비교해보세요!**
1. 여러 런을 체크박스로 선택
2. "Compare" 버튼 클릭
3. 파라미터-메트릭 차트에서 패턴 발견하기

## Part B: 커스텀 로깅 (20분)
autolog만으로 부족할 때 — 직접 기록하고 싶은 것들

In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import numpy as np

mlflow.autolog(disable=True)  # 커스텀 로깅을 위해 자동 로깅 끄기

with mlflow.start_run(run_name="custom_logging_demo"):
    # 1️⃣ 커스텀 파라미터 로깅
    params = {
        "n_estimators": 150,
        "max_depth": 7,
        "min_samples_split": 5,
        "data_version": "v1.0",  # 데이터 버전도 추적!
        "feature_count": X_train.shape[1],
    }
    mlflow.log_params(params)

    # 2️⃣ 모델 학습
    rf = RandomForestClassifier(**{k: v for k, v in params.items()
                                   if k in ["n_estimators", "max_depth", "min_samples_split"]},
                                random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    # 3️⃣ 커스텀 메트릭 로깅
    metrics = {
        "f1_score": f1_score(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
        "positive_ratio": y_test.mean(),
    }
    mlflow.log_metrics(metrics)

    # 4️⃣ 아티팩트 로깅 — 혼동행렬
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["보통", "좋음"])
    disp.plot(ax=ax, cmap="Blues")
    ax.set_title("Wine Quality Confusion Matrix")
    fig.savefig("/tmp/confusion_matrix.png", dpi=150, bbox_inches="tight")
    mlflow.log_artifact("/tmp/confusion_matrix.png")
    plt.close()

    # 5️⃣ 아티팩트 로깅 — ROC 커브
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color='#FF3621', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve')
    ax.legend()
    fig.savefig("/tmp/roc_curve.png", dpi=150, bbox_inches="tight")
    mlflow.log_artifact("/tmp/roc_curve.png")
    plt.close()

    # 6️⃣ 태그로 실험 분류
    mlflow.set_tags({
        "team": "data_school_3기",
        "task": "wine_quality_classification",
        "stage": "experiment",
    })

    # 7️⃣ 모델 로깅
    mlflow.sklearn.log_model(rf, "model")

    print(f"✅ 커스텀 로깅 완료!")
    print(f"   F1: {metrics['f1_score']:.4f}")
    print(f"   AUC: {roc_auc:.4f}")

/root/.ipykernel/1161/command-5154520214584130-3032168467:40: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from current font.
  fig.savefig("/tmp/confusion_matrix.png", dpi=150, bbox_inches="tight")
/root/.ipykernel/1161/command-5154520214584130-3032168467:40: UserWarning: Glyph 53685 (\N{HANGUL SYLLABLE TONG}) missing from current font.
  fig.savefig("/tmp/confusion_matrix.png", dpi=150, bbox_inches="tight")
/root/.ipykernel/1161/command-5154520214584130-3032168467:40: UserWarning: Glyph 51339 (\N{HANGUL SYLLABLE JOH}) missing from current font.
  fig.savefig("/tmp/confusion_matrix.png", dpi=150, bbox_inches="tight")
/root/.ipykernel/1161/command-5154520214584130-3032168467:40: UserWarning: Glyph 51020 (\N{HANGUL SYLLABLE EUM}) missing from current font.
  fig.savefig("/tmp/confusion_matrix.png", dpi=150, bbox_inches="tight")
2026/03/24 02:07:30 WARNING mlflow.models.model: Model logged without a signature. Signatures will be required for upcoming model registry features 

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

✅ 커스텀 로깅 완료!
   F1: 0.5455
   AUC: 0.8793


## Part C: 모델 시그니처 & 입력 예제 (10분)
모델의 "사용 설명서"를 함께 기록합니다.

In [0]:
from mlflow.models import infer_signature

mlflow.autolog(disable=True)

with mlflow.start_run(run_name="model_with_signature"):
    rf = RandomForestClassifier(n_estimators=150, max_depth=7, random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    # 시그니처 자동 추론: 입력/출력 스키마를 모델에 기록
    signature = infer_signature(X_train, y_pred)

    # 입력 예제: 모델 테스트에 사용할 샘플 데이터
    input_example = X_test.head(3)

    mlflow.sklearn.log_model(
        rf,
        "model",
        signature=signature,
        input_example=input_example,
    )

    print("✅ 시그니처 포함 모델 로깅 완료!")
    print(f"\n📋 시그니처 정보:")
    print(signature)

Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

✅ 시그니처 포함 모델 로깅 완료!

📋 시그니처 정보:
inputs: 
  ['fixed_acidity': double (required), 'volatile_acidity': double (required), 'citric_acid': double (required), 'residual_sugar': double (required), 'chlorides': double (required), 'free_sulfur_dioxide': double (required), 'total_sulfur_dioxide': double (required), 'density': double (required), 'pH': double (required), 'sulphates': double (required), 'alcohol': double (required)]
outputs: 
  [Tensor('int32', (-1,))]
params: 
  None



In [0]:
# 저장된 모델 로드 & 예측 테스트
run_id = mlflow.last_active_run().info.run_id
loaded_model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")

# 시그니처 확인
print("모델 시그니처:", loaded_model.metadata.signature)

# 예측 테스트
sample = X_test.head(5)
predictions = loaded_model.predict(sample)
print(f"\n예측 결과: {predictions}")

모델 시그니처: inputs: 
  ['fixed_acidity': double (required), 'volatile_acidity': double (required), 'citric_acid': double (required), 'residual_sugar': double (required), 'chlorides': double (required), 'free_sulfur_dioxide': double (required), 'total_sulfur_dioxide': double (required), 'density': double (required), 'pH': double (required), 'sulphates': double (required), 'alcohol': double (required)]
outputs: 
  [Tensor('int32', (-1,))]
params: 
  None


예측 결과: [0 1 1 0 1]


## 🎯 핵심 정리

| 기능 | 코드 | 용도 |
|------|------|------|
| **자동 로깅** | `mlflow.autolog()` | 빠른 실험, 기본 추적 |
| **파라미터** | `mlflow.log_param(key, value)` | 하이퍼파라미터, 설정값 |
| **메트릭** | `mlflow.log_metric(key, value)` | 성능 수치 |
| **아티팩트** | `mlflow.log_artifact(path)` | 차트, 데이터 파일 |
| **태그** | `mlflow.set_tag(key, value)` | 실험 분류, 메타데이터 |
| **시그니처** | `infer_signature(X, y)` | 모델 입출력 스키마 |

---
**다음 실습**: 실습 4 — Feature Store & 배치 추론